# 06 - Exportar a TFLite

Convierte M1/M2 (doble entrada) y M_seg (hoja/fondo) a `.tflite` (float32 + int8). Genera labels.

In [ ]:
!pip install -q tensorflow opencv-python-headless

In [ ]:
import json
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path
from PIL import Image
from tensorflow.keras.applications.efficientnet import preprocess_input

OUT = Path('./outputs'); SPLIT = Path('./splits')
m1 = tf.keras.models.load_model(OUT / 'model1_binary.keras', compile=False)
m2 = tf.keras.models.load_model(OUT / 'model2_pathogen.keras', compile=False)
mseg = tf.keras.models.load_model(OUT / 'model_seg.keras', compile=False)
print('modelos cargados')

_HSV_GREEN_LOW = np.array([20, 25, 30]); _HSV_GREEN_HIGH = np.array([90, 255, 255])
_HSV_LESION_LOW = np.array([0, 40, 25]); _HSV_LESION_HIGH = np.array([35, 255, 230]); _MK = np.ones((7, 7), np.uint8)
def _leaf_iso(img):
    hsv = cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    m = cv2.bitwise_or(cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH), cv2.inRange(hsv, _HSV_LESION_LOW, _HSV_LESION_HIGH))
    m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_CLOSE, _MK), cv2.MORPH_OPEN, _MK)
    o = img.copy(); o[m == 0] = 0; return o
def _norm(img):
    r = img.astype(np.float32); a = r.reshape(-1, 3).mean(0); return np.clip(r * (a.mean() / (a + 1e-6)), 0, 255).astype(np.uint8)

In [ ]:
def _cls_samples(directory, size, n=80):
    paths = []
    for c in sorted(p for p in Path(directory).iterdir() if p.is_dir()):
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            paths += list(c.glob(ext))
    paths = paths[:n]
    for fp in paths:
        img = np.array(Image.open(fp).convert('RGB').resize(size))
        yield preprocess_input(img.astype(np.float32)), preprocess_input(_leaf_iso(img).astype(np.float32))


def export_dual(model, name, directory, size):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    (OUT / f'{name}.tflite').write_bytes(conv.convert())
    samples = list(_cls_samples(directory, size))
    def rep():
        for o, l in samples:
            yield [o[np.newaxis], l[np.newaxis]]
    c8 = tf.lite.TFLiteConverter.from_keras_model(model)
    c8.optimizations = [tf.lite.Optimize.DEFAULT]
    c8.representative_dataset = rep
    c8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c8.inference_input_type = tf.uint8; c8.inference_output_type = tf.uint8
    (OUT / f'{name}_int8.tflite').write_bytes(c8.convert())
    print(name, 'float32', round((OUT / f'{name}.tflite').stat().st_size / 1e6, 2), 'MB | int8', round((OUT / f'{name}_int8.tflite').stat().st_size / 1e6, 2), 'MB')


export_dual(m1, 'model1', SPLIT / 'clasificacion_binaria' / 'val', (240, 240))
export_dual(m2, 'model2', SPLIT / 'clasificacion_patogeno' / 'val', (224, 224))

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(mseg)
(OUT / 'model_seg.tflite').write_bytes(conv.convert())

def _seg_rep():
    d = SPLIT / 'clasificacion_patogeno' / 'val'
    paths = []
    for c in sorted(p for p in d.iterdir() if p.is_dir()):
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            paths += list(c.glob(ext))[:12]
    for fp in paths[:80]:
        img = _norm(np.array(Image.open(fp).convert('RGB').resize((256, 256))))
        yield [(img.astype(np.float32) / 255.0)[np.newaxis]]

c8 = tf.lite.TFLiteConverter.from_keras_model(mseg)
c8.optimizations = [tf.lite.Optimize.DEFAULT]
c8.representative_dataset = _seg_rep
c8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c8.inference_input_type = tf.uint8; c8.inference_output_type = tf.uint8
(OUT / 'model_seg_int8.tflite').write_bytes(c8.convert())
print('model_seg float32', round((OUT / 'model_seg.tflite').stat().st_size / 1e6, 2), 'MB | int8', round((OUT / 'model_seg_int8.tflite').stat().st_size / 1e6, 2), 'MB')

In [ ]:
def labels_from(idx_path, out_path):
    idx = json.load(open(idx_path))
    labels = [k for k, _ in sorted(idx.items(), key=lambda kv: kv[1])]
    Path(out_path).write_text('\n'.join(f'{i} {l}' for i, l in enumerate(labels)))
    print(out_path, labels)

labels_from(OUT / 'class_indices_model1_binary.json', OUT / 'labels_m1.txt')
labels_from(OUT / 'class_indices_model2_pathogen.json', OUT / 'labels_m2.txt')